# JIN AI — Phase 17: continued pretraining on Kaggle

**Read this first.** This notebook continues the Phase 14 checkpoint. It does **not**
start a new model, and it does not promise a capability gain: continued pretraining
improves language modelling, and whether that becomes task ability is measured, not
assumed.

Sessions are bounded by wall clock (`--session-max-minutes`); the global step budget
never changes between sessions. If you need a bigger budget, start a new documented
experiment with a new parent SHA.

Every stage prints what it found before it acts. Nothing assumes a GPU model, VRAM
size, runtime quota or dataset slug.

## Stage B — configuration (edit this cell only)

Kaggle mounts inputs read-only under `/kaggle/input/<dataset-slug>/`. Fill in the
paths you actually attached; the notebook never guesses a slug.

In [ ]:
import os, sys, json, pathlib, shutil, subprocess

CONFIG = {
    # 1. Public GitHub repo (attach as a Kaggle Dataset, or clone).
    "repo_path": "/kaggle/input/jin-ai-repo/JIN-AI-LLM-from-zero-to-hero-ZHIEND",
    # 2. Phase 14 checkpoint, attached as its own dataset (not in the repo).
    "checkpoint_path": "/kaggle/input/jin-phase14-checkpoint/best.pt",
    # 3. Wikipedia parquet, attached as a dataset directory.
    "wikipedia_path": "/kaggle/input/wikipedia-en-parquet",
    # 4. Optional: corpus prepared by an earlier session.
    "corpus_v7_path": None,
    # 5. Optional: checkpoint from the previous Phase 17 session.
    "phase17_ckpt_path": None,
    "out_dir": "/kaggle/working/phase17_output",
}

REPO = CONFIG["repo_path"]
OUT = CONFIG["out_dir"]
os.makedirs(OUT, exist_ok=True)
print(json.dumps(CONFIG, indent=2))

## Stage A — environment preflight

Detects GPU, VRAM, CUDA, torch, CPU, RAM, disk, and whether the checkpoint and data
are actually present. Fails loudly if the checkpoint is missing: continued
pretraining must never fall back to random weights.

In [ ]:
!nvidia-smi || echo 'no nvidia-smi'
os.chdir(REPO)
subprocess.run([sys.executable, "scripts/kaggle_preflight.py",
                "--checkpoint", CONFIG["checkpoint_path"],
                "--out", f"{OUT}/kaggle_preflight.json"], check=True)
preflight = json.load(open(f"{OUT}/kaggle_preflight.json"))
print("single-GPU trainer:", preflight["torch"]["device_count"], "device(s) visible,",
      "the training loop uses device 0 only")

## Stage C — corpus preflight (do not run `--all` blindly)

Lists discovered sources, resolves the Wikipedia source id through the reviewed
licence mapping, and estimates size from a bounded sample.

In [ ]:
!python scripts/ingest_corpus.py --list
!python - <<'PY'
import sys; sys.path.insert(0, 'scripts')
from license_policy import classify_source
for sid in ('wikipedia_en_20231101', 'wikipedia-en-20231101'):
    d = classify_source(sid, None)
    print(f'{sid:24s} -> {d.canonical:24s} eligible={d.eligible}  {d.reason}')
PY

In [ ]:
# Bounded sample first: 10k documents is enough to project disk and memory.
!python scripts/ingest_corpus.py --only wikipedia_en_20231101 \
    --max-docs-per-source 10000 --out-root data/curated_v7_sample
!du -sh data/curated_v7_sample
print('Project full size before committing to a 10 GB ingest.')

## Stage D — corpus v7 (only after Stage C projections fit)

These are the exact commands; run them one at a time and read the audit before
moving on. `--pii-policy exclude --license-policy enforce` are not optional.

In [ ]:
# 1) Ingest (source identity, revision, licence, PII findings preserved)
# !python scripts/ingest_corpus.py --only wikipedia_en_20231101 --max-docs-per-source 1000000 --out-root data/curated_v7

# 2) Verify (dedup, splits, licence, PII) — inspect reports/dedup_quality_splits.json
# !python scripts/verify_corpus.py --in-root data/curated_v7 --out-root data/curated_v7 --pii-policy exclude --license-policy enforce

# 3) Tokenise with an explicit mixture (weights are targets, not guarantees)
# !python scripts/build_tokenized_corpus.py --splits data/curated_v7/splits --out data/gpt2_corpus_v7 \
#     --max-train-tokens 1000000000 --mix wikipedia_en_20231101:0.65,cosmopedia_v2_sample:0.25,prepared_v4:0.08,project_seed:0.02
print('Stage D pending: check Stage C numbers first.')

## Stage E — smoke test from the Phase 14 weights (never random init)

20 steps at context 1024. The first validation loss must be near the parent's
(~4.7), not near ln(50257) ≈ 10.8. If it is near 10.8, the weights were not
transferred and the run must stop.

In [ ]:
!python train_gpt2.py --preset gpt2_small_reference --data-dir data/gpt2_corpus_v5 \
    --init-from {CONFIG['checkpoint_path']} --continue-pretrain \
    --out-dir {OUT}/smoke --seq-len 1024 --batch-size 4 --grad-accum 4 \
    --max-steps 20 --warmup 2 --lr 1e-4 --eval-interval 10 --eval-iters 4 \
    --save-interval 10 --seed 17

In [ ]:
m = json.load(open(f"{OUT}/smoke/latest.pt.manifest.json"))
assert m["init_mode"] == "continued_pretrain", m["init_mode"]
assert m["parent_sha256"], "parent SHA missing"
print(json.dumps({k: m[k] for k in ("iteration", "tokens_seen", "init_mode",
                                    "parent_sha256", "additional_tokens_this_run")}, indent=2))

## Stage F — learning-rate pilot (do not reuse 6e-4 untested)

Same parent, same corpus, same seed, same geometry. Three candidates, each bounded
to a few hundred steps. Choose on stability and clean validation, not on the lowest
training loss.

In [ ]:
for lr in ("3e-5", "1e-4", "3e-4"):
    print(f"--- lr {lr} ---")
    !python train_gpt2.py --preset gpt2_small_reference --data-dir data/gpt2_corpus_v5 \
        --init-from {CONFIG['checkpoint_path']} --continue-pretrain \
        --out-dir {OUT}/lr_pilot_{lr} --seq-len 1024 --batch-size 4 --grad-accum 8 \
        --max-steps 300 --warmup 20 --lr {lr} --eval-interval 100 --eval-iters 4 \
        --save-interval 100 --session-max-minutes 25 --seed 17

## Stage G — continued pretraining, one Kaggle session

Global budget for the first substantial milestone: 100M additional tokens.
With batch 4 × accum 8 × 1024 = 32,768 tokens/step that is about 3,052 steps.
The number below is computed, not guessed: change `--max-steps` only by editing
`TOTAL_STEPS` here and keep it fixed for every later session.

In [ ]:
TOKENS_PER_STEP = 4 * 8 * 1024
TARGET_ADDITIONAL_TOKENS = 100_000_000
TOTAL_STEPS = TARGET_ADDITIONAL_TOKENS // TOKENS_PER_STEP
print(TOTAL_STEPS, "steps x", TOKENS_PER_STEP, "tokens =", TOTAL_STEPS * TOKENS_PER_STEP)

# First session: start from Phase 14.
!python train_gpt2.py --preset gpt2_small_reference --data-dir data/gpt2_corpus_v5 \
    --init-from {CONFIG['checkpoint_path']} --continue-pretrain \
    --out-dir {OUT}/run_100m --seq-len 1024 --batch-size 4 --grad-accum 8 \
    --max-steps {TOTAL_STEPS} --warmup 100 --lr 1e-4 --eval-interval 250 \
    --eval-iters 8 --save-interval 250 --session-max-minutes 500 --seed 17

## Stage G2 — next sessions: resume the same experiment

Attach the previous session's output (or the exported artifact) and resume. The
step budget must be identical to the first session, otherwise the run is refused:
a different budget is a different schedule, not a continuation of this one.

In [ ]:
PREV = CONFIG["phase17_ckpt_path"]  # e.g. /kaggle/input/jin-phase17-s1/latest.pt
if PREV:
    !python train_gpt2.py --preset gpt2_small_reference --data-dir data/gpt2_corpus_v5 \
        --resume --resume-from {PREV} --out-dir {OUT}/run_100m \
        --seq-len 1024 --batch-size 4 --grad-accum 8 \
        --max-steps {TOTAL_STEPS} --warmup 100 --lr 1e-4 --eval-interval 250 \
        --eval-iters 8 --save-interval 250 --session-max-minutes 500 --seed 17
else:
    print('Set CONFIG["phase17_ckpt_path"] to the previous session latest.pt')

## Stage H — evaluation and export

Language modelling is measured with the frozen Phase 16 protocol; capability is
measured with the frozen task benchmark. A lower loss is not a capability claim.

In [ ]:
CKPT = f"{OUT}/run_100m/best.pt"
!python scripts/eval_phase16.py --checkpoint {CKPT} \
    --val-jsonl data/curated_v6/benchmark/clean_unseen.jsonl --seq-len 1024 \
    --out {OUT}/benchmark_results/eval_clean.json
!python scripts/phase16_task_benchmark.py --checkpoint {CKPT} \
    --out {OUT}/benchmark_results/task_benchmark.json

In [ ]:
import hashlib
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for c in iter(lambda: fh.read(1 << 22), b''):
            h.update(c)
    return h.hexdigest()

man = json.load(open(f"{OUT}/run_100m/latest.pt.manifest.json"))
actual = sha(f"{OUT}/run_100m/latest.pt")
assert actual == man["sha256"], "checkpoint hash does not match its manifest"
print(json.dumps({"sha256": actual, "iteration": man["iteration"],
                  "additional_tokens": man["additional_tokens_this_run"],
                  "parent_sha256": man["parent_sha256"]}, indent=2))
print("\nNext session needs these files: latest.pt, latest.pt.manifest.json, session_log.json")
print("Save this notebook version (or 'Save & Run All') to publish the output as a Dataset,")
print("then attach that Dataset as an input for the next session.")

## Recovery

* Session died before a save → the previous `latest.pt` is intact (atomic writes); rerun Stage G2 with the same `TOTAL_STEPS`.
* Refused with "Training configuration changed" → something in the recipe differs; do not force it, compare `experiment_config` in the two manifests.
* Refused with "Step budget changed" → you changed `--max-steps`; use the original value, or start a new documented experiment.
* First smoke loss near 10.8 → weights were not transferred; check `--init-from` and the parent SHA before spending more GPU time.